# Read me

In this notebook, we use a Autoencoder with a NearestNeighbors model with Euclidean distance.

9 features → Encoder → 4 embeddings → k-NN → 20 recommandations

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

from keras import Sequential, Input, Model
from keras.layers import Dense
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping

from IPython.display import HTML, display

import joblib

# Import Spotify module
import app.spotify.spotify as spo
from app.preprocessing.preprocessing import preprocessing

In [14]:
# features
audio_features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "valence",
    "tempo"
]

# hyper parameters
latent_dimension = 4
input_dimension = 8
hidden_dimension = 16

# Spotify access token
#token = spo.get_access_token()


# Load data

In [4]:
df = pd.read_csv("../raw_data/spotify-tracks-dataset-detailed.csv")
print(f'Loaded {len(df)} rows')

Loaded 114000 rows


# Preprocessing

In [5]:
X_scaled, df_metadata, scaler, features = preprocessing(df)

print(f'Loaded {len(X_scaled)} rows')

Loaded 80175 rows


In [10]:
X_train_scaled, X_temp = train_test_split(
    X_scaled, test_size=0.30, random_state=42
)

X_val_scaled, X_test_scaled = train_test_split(
    X_temp, test_size=0.50, random_state=42
)

In [24]:
joblib.dump(scaler, "../artifacts/models/autoencoder_knn/autoencoder_knn_scaler.joblib")

['../artifacts/models/autoencoder_knn/autoencoder_knn_scaler.joblib']

# Pre-process data
**Deprecated. Use preprocessing instead.**

1. Remove Comedy
2. Drop duplicates
3. Drop missing values

In [5]:
len_before = len(df)
df = df[df["track_genre"] != "comedy"].reset_index(drop=True)
len_after = len(df)

print(f'Removed {len_before - len_after} comedy tracks. Left {len_after} rows')

len_before = len(df)
df = df.drop_duplicates(subset=["track_id"]).drop_duplicates(subset=["artists", "track_name"]).reset_index(drop=True)
len_after = len(df)

print(f'Removed {len_before - len_after} duplicates. Left {len_after} rows')

len_before = len(df)
df = df.dropna(subset=['album_name']).dropna(subset=['track_name']).reset_index(drop=True)
len_after = len(df)

print(f'Removed {len_before - len_after} null values. Left {len(df)} rows')

Removed 1000 comedy tracks. Left 113000 rows
Removed 32642 duplicates. Left 80358 rows
Removed 1 null values. Left 80357 rows


# Split Train Val Test
**Deprecated. Use preprocessing instead.**

In [6]:
X = df[audio_features]

X_train, X_temp = train_test_split(
    X, test_size=0.30, random_state=42
)

X_val, X_test = train_test_split(
    X_temp, test_size=0.50, random_state=42
)

# Scaling
**Deprecated. Use preprocessing instead.**

In [7]:
scaler = StandardScaler().set_output(transform="pandas")

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Build autoencoder
1. Build encoder
2. Build decoder
3. Stack them together

In [15]:
def build_encoder(input_dimension, hidden_dimension, latent_dimension):
    """
    Returns an encoder model
    """

    encoder = Sequential(name="encoder")

    encoder.add(Input(shape=(input_dimension,)))
    encoder.add(Dense(hidden_dimension, activation='relu'))
    encoder.add(Dense(latent_dimension, activation='linear'))

    return encoder


def build_decoder(latent_dimension, hidden_dimension, input_dimension=9):
    """
    Returns an decoder model
    """

    decoder = Sequential(name="decoder")

    decoder.add(Input(shape=(latent_dimension,)))
    decoder.add(Dense(hidden_dimension, activation="relu"))
    decoder.add(Dense(input_dimension, activation="linear"))

    return decoder


encoder = build_encoder(input_dimension, hidden_dimension, latent_dimension)
decoder = build_decoder(latent_dimension, hidden_dimension, input_dimension)

inputs = Input(shape=(8,))
outputs = decoder(encoder(inputs))

autoencoder = Model(inputs, outputs)

autoencoder.compile(
    optimizer=Adam(
        learning_rate=0.001
    ),
    loss="mse"
)

autoencoder.summary()

Model: "model_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_15 (InputLayer)       [(None, 8)]               0         
                                                                 
 encoder (Sequential)        (None, 4)                 212       
                                                                 
 decoder (Sequential)        (None, 8)                 216       
                                                                 
Total params: 428 (1.67 KB)
Trainable params: 428 (1.67 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


# Train

In [16]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = autoencoder.fit(
    X_train_scaled,
    X_train_scaled,
    validation_data=(
        X_val_scaled,
        X_val_scaled
    ),
    epochs=100,
    batch_size=256,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/100
220/220 [==============================] - 1s 1ms/step - loss: 0.5244 - val_loss: 0.3004
Epoch 2/100
220/220 [==============================] - 0s 376us/step - loss: 0.2485 - val_loss: 0.2091
Epoch 3/100
220/220 [==============================] - 0s 345us/step - loss: 0.1886 - val_loss: 0.1755
Epoch 4/100
220/220 [==============================] - 0s 350us/step - loss: 0.1653 - val_loss: 0.1579
Epoch 5/100
220/220 [==============================] - 0s 417us/step - loss: 0.1503 - val_loss: 0.1447
Epoch 6/100
220/220 [==============================] - 0s 380us/step - loss: 0.1386 - val_loss: 0.1347
Epoch 7/100
220/220 [==============================] - 0s 360us/step - loss: 0.1302 - val_loss: 0.1273
Epoch 8/100
220/220 [==============================] - 0s 351us/step - loss: 0.1234 - val_loss: 0.1202
Epoch 9/100
220/220 [==============================] - 0s 348us/step - loss: 0.1165 - val_loss: 0.1134
Epoch 10/100
220/220 [==============================] - 0s 339us/step - los

In [18]:
# save encoder
encoder.save('../artifacts/models/autoencoder_knn/autoencoder_encoder.keras')

# Encode X into embedings

In [20]:
# Encode the tracks

#X_scaled = scaler.transform(X)
X_embeddings = encoder.predict(X_scaled, verbose=0)

print(X_scaled.shape)
print(X_embeddings.shape)

(80175, 8)
(80175, 4)


# Train k-NN model with embedings

In [21]:
# NearestNeighbors with Euclidean distance
model = NearestNeighbors(metric="euclidean")

model.fit(X_embeddings)

,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'euclidean'
,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'eu...an'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


In [22]:
# save model

joblib.dump(model, "../artifacts/models/autoencoder_knn/autoencoder_knn_model.joblib")

['../artifacts/models/autoencoder_knn/autoencoder_knn_model.joblib']

# Build input tracks list

In [4]:
# Build an input track set of 10 tracks
genre = "jazz"

input_tracks = df[df["track_genre"] == genre].sample(
    n=10,
    random_state=42
)

In [ ]:
# input_tracks.to_csv('jazz.csv', index=False)

In [44]:
input_tracks[['track_id', 'artists', 'album_name', 'track_name', 'track_genre']]

,track_id,artists,album_name,track_name,track_genre
48644,0Oho1Elt6nNyiHZyZR8OaG,Jane Monheit,Surrender,A Time For Love,jazz
48653,7esh5OJ7CP435rmXeGm305,Eartha Kitt,Classic Christmas Greatest Hits,Santa Baby,jazz
48577,5cQANJMHoDKb2fCxr5pZkq,Leo Gandelman;Baco Exu do Blues;Machine Series,Hip Hop Machine #6,Flamingos,jazz
48616,4nbILrYODP667b3d66ZMzI,Dean Martin,"Hey, Brother Pour The Wine",Sway (Quien Sera),jazz
48454,1fBUmT2qNGdTgaS4XXwGnA,Nat King Cole,Christmas Only: The Timeless Classic Collection,O Little Town of Bethlehem,jazz
48550,13cpOnmccZGVGFngLxYN4I,Kenny G,Breathless,Forever in Love,jazz
48621,4m6OC8u2rC6LMFtFWsME7d,Chet Baker,Chet Baker Sings,Time After Time - Vocal Version,jazz
48512,6SvIO9X1alZpKyG3g8EAHB,Peggy Lee,Christmas Classics,The Christmas Waltz,jazz
48645,3Nsn3y2IpzcuDIKZDgS3oS,Luis Arcaraz Y Su Orquesta,Cita Con Luis Arcaraz,Viajera,jazz
48578,4v5PKC5rvQwb33NWX82UH8,Carlos Lyra,Bossa Nova Carlos Lyra,Você e Eu (You and Me),jazz


In [40]:
# Get input track indices
input_indices = input_tracks.index.to_list()

In [37]:
playlist_id = spo.create_playlist(token)
spo.add_tracks_to_playlist(token, playlist_id, df.iloc[input_indices]['track_id'])
spo.display_player(playlist_id)

# Recommend

In [45]:
# Calculate input profile
input_profile = X_embeddings[input_indices].mean(axis=0).reshape(1, -1)

# Find 20 neighbors
distances, indices = model.kneighbors(input_profile, n_neighbors=20)

In [ ]:
# Show player
playlist_id = spo.create_playlist(token)
spo.add_tracks_to_playlist(token, playlist_id, df.iloc[indices[0]]['track_id'])
spo.display_player(playlist_id)

In [43]:
# The model returns an array of indices
# Return the tracks from the original dataframe
recommendations = df.iloc[indices[0]]
recommendations[['track_id', 'artists', 'album_name', 'track_name', 'track_genre']]


,track_id,artists,album_name,track_name,track_genre
42892,0xHvmWCpz9Pnn1sDvRehAs,Pritam;Arijit Singh,Ae Dil Hai Mushkil (Original Motion Picture So...,Channa Mereya - Unplugged,indian
66100,24fsdEtEb9U7cwlPvbc5xr,Nikolay Kopylov,"Ради Женщин (Сцены, Арии И Дуэты Из Классическ...",Ария Раджами Из Оперетты «Баядера»,romance
26760,0VxocBntP1XTZRsR9ZURPS,Daniela Andrade,La Vie En Rose,La Vie En Rose,folk
10321,7jFNX9aTdcQfhIgUVE02hi,Hacken Lee,命運符號,月半小夜曲,cantopop
65228,4rkv812lQ9utiRQ9EMLIKb,The Platters,The Very Best Of The Platters,I'm Sorry,rockabilly
438,68KCwQ2PNKNT17sEx32DEG,The Civil Wars,Barton Hollow,Dance Me to the End of Love - Bonus,acoustic
13309,3r9eGg4SVPbhtiPTA9UkmK,keshi,always,us,chill
58775,7jKCAmjP2cAn9WxxIVR14Y,Skylar Grey,Aquaman (Original Motion Picture Soundtrack) [...,Everything I Need,piano
3369,2kNwTOVdZGCGROgAzjD1J0,Julien Baker,Guthrie,Guthrie,ambient
10415,1SjCVgrZqywAgwe0U8pGJk,Eason Chan,The Line-Up,人來人往,cantopop
